Script to compare GOGET locations to Gov data locations. Please run this code in playground mode.

In [44]:
# import libraries

import geopandas
import shapely.geometry
import shapely.ops
import pyproj
import openpyxl
import pandas as pd

import time

In [45]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [46]:
# load in files
GOGET_Iteration_Folder = '/content/drive/Shareddrives/GEM Shared Drive/Programs/Oil & Gas Program/Global Oil and Gas Extraction Tracker (GOGET)/GOGET Iterations/Snapshots of GOGET main data set/'
GOGET_Iteration = GOGET_Iteration_Folder + 'GOGET oil & gas extraction sites (2025 07 08).xlsx' #change this with the interation you want to us


In [47]:
# set parameters
country_of_interest = 'Brazil' # change this with the country you are working on. Make sure the name is exactly what is used in GOGET and in Rystad.
# add handling of mistmatch name
now = time.localtime()

# create timestamp
timestamp = time.strftime("%Y%m%d_%H%M", now)

In [66]:
# import field data
Field_data = GOGET_Iteration

# convert csv to gdf, creating geometry from longitude and latitude
# First, read the CSV file into a pandas DataFrame
Fielddata_df = pd.read_excel(GOGET_Iteration, sheet_name='Main data', engine='openpyxl')

# filter for country of interest
Fielddata_df = Fielddata_df[Fielddata_df['Country/Area'] == country_of_interest]

# Clean the 'Longitude' and 'Latitude' columns by replacing empty strings with NaN
Fielddata_df['Longitude'] = pd.to_numeric(Fielddata_df['Longitude'], errors='coerce')
Fielddata_df['Latitude'] = pd.to_numeric(Fielddata_df['Latitude'], errors='coerce')

# Now, convert the cleaned pandas DataFrame to a GeoDataFrame
Fielddata_gdf = geopandas.GeoDataFrame(
    Fielddata_df,
    geometry=geopandas.points_from_xy(Fielddata_df.Longitude,
                                      Fielddata_df.Latitude)
)
# Set the CRS for Fielddata_gdf
Fielddata_gdf = Fielddata_gdf.set_crs(epsg=4326)

# Drop rows with invalid geometries
Fielddata_gdf = Fielddata_gdf[Fielddata_gdf.geometry.is_valid]

KeyboardInterrupt: 

In [67]:
Fielddata_gdf

,Unit ID,Unit Name,Unit name local script,Country/Area,"Subnational unit (province, state)",Status,Status year,Status source,Latitude,Longitude,...,Wiki URL,Wiki URL local,Fuel type,Unit type,Production data?,Reserves data?,Last updated,Notes,keep,geometry
1,OG0000002,Abalone,Abalone,Brazil,Espírito Santo,operating,2024,ANP Shapefile de Dados 2024,-21.360000,-39.612000,...,https://www.gem.wiki/Abalone_Oil_and_Gas_Field...,https://www.gem.wiki/Campo_de_Petróleo_e_Gás_A...,oil and gas,field,2023 / 2023 /,2015,2024-11-21,Fields in this block are going to be decomissi...,yes,POINT (-39.612 -21.36)
2,OG0000003,Acajá-Burizinho,Acajá-Burizinho,Brazil,Bahia,operating,2024,ANP Shapefile de Dados 2024,-12.259000,-38.076000,...,NaN,NaN,oil and gas,field,2023 / 2023 /,2014,2024-11-21,NaN,NaN,POINT (-38.076 -12.259)
3,OG0000004,Acauã,Acauã,Brazil,Rio Grande do Norte,operating,2024,ANP Shapefile de Dados 2024,-5.269000,-37.366000,...,NaN,NaN,oil and gas,field,2023 / 2023 /,2015,2024-11-21,NaN,NaN,POINT (-37.366 -5.269)
4,OG0000005,Acauã Leste,Acauã Leste,Brazil,Rio Grande do Norte,discovered,2019,ANP Stat 2020 Frame 1.2,-5.265370,-37.326490,...,NaN,NaN,oil and gas,field,no data - no entry on production & reserve sheet,no data - no entry on production & reserve sheet,2024-11-21,NaN,NaN,POINT (-37.32649 -5.26537)
5,OG0000006,Aguilhada,Aguilhada,Brazil,Sergipe,operating,2023,BMPPGN2023,-10.686000,-36.889000,...,https://www.gem.wiki/Aguilhada_Oil_and_Gas_Fie...,https://www.gem.wiki/Campo_de_Petróleo_e_Gás_A...,oil and gas,field,2023 / 2023 /,2015,2021-02-01,NaN,yes,POINT (-36.889 -10.686)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13258,OG0021317,Irara,Irara,Brazil,Espírito Santo,operating,2023,https://www.gov.br/anp/pt-br/assuntos/explorac...,-19.271411,-39.819201,...,NaN,NaN,oil,field,2023 / 2023 /,no data,2024-11-29,NaN,NaN,POINT (-39.8192 -19.27141)
13259,OG0021318,Suindara,Suindara,Brazil,Espírito Santo,operating,2023,https://www.gov.br/anp/pt-br/assuntos/explorac...,-19.198208,-39.836095,...,NaN,NaN,oil,field,2023 / 2023 /,no data,2024-11-29,NaN,NaN,POINT (-39.8361 -19.19821)
13263,OG0021322,Mero (non-contracted area),Mero (Área não contratada),Brazil,Rio de Janeiro,operating,2023,BMPPGN2023,-24.612000,-42.220000,...,https://www.gem.wiki/Mero_(non-contracted_area...,NaN,oil,field,2023 / 2023 /,no data,2024-12-02,NaN,yes,POINT (-42.22 -24.612)
13265,OG0021324,Forno (non-contracted area),Forno (Área não contratada),Brazil,Rio de Janeiro,operating,2023,BMPPGN2023,-22.131000,-39.961000,...,NaN,NaN,oil and gas,field,2023 / /,no data,2024-12-03,NaN,NaN,POINT (-39.961 -22.131)


In [68]:
# define folders
goget_country_folder = '/content/drive/Shareddrives/GEM Shared Drive/Programs/Oil & Gas Program/Global Oil and Gas Extraction Tracker (GOGET)/Data sets & reports (O&G)/O&G country materials/'
country = 'Brazil (GOGET O&G)/' # change for country you are working on

In [78]:
# import comparison data
Comparison_folder = goget_country_folder + country
Comparison_file = 'Brazil fields with coord and wkt - Location Comparison Sheet.csv'

# read in data
compdf = pd.read_csv(Comparison_folder + Comparison_file)

# Filter and convert 'Longitude' and 'Latitude' using .loc
compdf.loc[:, 'Longitude'] = pd.to_numeric(compdf.loc[:, 'Longitude'], errors='coerce')
compdf.loc[:, 'Latitude'] = pd.to_numeric(compdf.loc[:, 'Latitude'], errors='coerce')

# Now, convert the cleaned pandas DataFrame to a GeoDataFrame
compgdf = geopandas.GeoDataFrame(
    compdf,
    geometry=geopandas.points_from_xy(compdf.Longitude,
                                      compdf.Latitude)
)
# Set the CRS for compgdf
compgdf = compgdf.set_crs(epsg=4326)

# Drop rows with invalid geometries
compgdf = compgdf[compgdf.geometry.is_valid]

In [79]:
compgdf

,Field Name,Field Code,Field ID,Latitude,Longitude,geometry
0,ABALONE,ABA,1843.0,-21.360,-39.612,POINT (-39.612 -21.36)
1,ACAJ�-BURIZINHO,AJBZ,1184.0,-12.259,-38.076,POINT (-38.076 -12.259)
2,ACAU�,AC,1380.0,-5.269,-37.366,POINT (-37.366 -5.269)
3,AGUILHADA,AG,1188.0,-10.686,-36.889,POINT (-36.889 -10.686)
4,AGULHA,AG,1189.0,-4.899,-36.262,POINT (-36.262 -4.899)
...,...,...,...,...,...,...
422,WAHOO,WAH,2360.0,-21.649,-39.863,POINT (-39.863 -21.649)
423,XAR�U,XA,981.0,-3.032,-39.035,POINT (-39.035 -3.032)
424,XISTO S�O MATEUS DO SUL,XSMS,2600.0,-25.804,-50.458,POINT (-50.458 -25.804)
425,�GUA GRANDE,AG,1187.0,-12.371,-38.344,POINT (-38.344 -12.371)


In [80]:
# Find closest assets in Fielddata_df to compgdf
def find_closest_assets(Fielddata_gdf, compgdf):
    """
    Finds the closest assets in Fielddata_gdf to each asset in compgdf.

    Args:
        Fielddata_gdf: GeoDataFrame of field data assets.
        compgdf: GeoDataFrame of comparison assets.

    Returns:
        pandas.DataFrame: DataFrame with closest asset information.
    """

    closest_assets = []
    for index, row in compgdf.iterrows():
        distances = Fielddata_gdf.distance(row['geometry'])
        min_distance = distances.min()
        closest_asset_index = distances.idxmin()
        closest_asset = Fielddata_gdf.loc[closest_asset_index]
        closest_assets.append({
            'comp_asset_index': index,
            'comp_longitude': row['Longitude'],
            'comp_latitude': row['Latitude'],
            'BrazilGov Name': row['Field Name'],  # Include Govdata Name
            'BrazilGov Field ID': row['Field ID'],  # Include Govdata Field ID
            'closest_asset_index': closest_asset_index,
            'closest_asset_longitude': closest_asset['Longitude'],
            'closest_asset_latitude': closest_asset['Latitude'],
            'Unit Name': closest_asset['Unit Name'],  # Include Unit Name
            'distance': min_distance
        })

    return pd.DataFrame(closest_assets)

# Apply function
Brazil_to_goget_assets_df = find_closest_assets(Fielddata_gdf, compgdf)

/tmp/ipython-input-1991154031.py:16: UserWarning: Geometry is in a geographic CRS. Results from 'distance' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  distances = Fielddata_gdf.distance(row['geometry'])
/tmp/ipython-input-1991154031.py:16: UserWarning: Geometry is in a geographic CRS. Results from 'distance' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  distances = Fielddata_gdf.distance(row['geometry'])
/tmp/ipython-input-1991154031.py:16: UserWarning: Geometry is in a geographic CRS. Results from 'distance' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  distances = Fielddata_gdf.distance(row['geometry'])
/tmp/ipython-input-1991154031.py:16: UserWarning: Geometry is in a geographic CRS. Results from 'distance' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geom

In [76]:
# sort by distance, low to high
Brazil_to_goget_assets_df = Brazil_to_goget_assets_df.sort_values(by='distance')

In [81]:
Brazil_to_goget_assets_df

,comp_asset_index,comp_longitude,comp_latitude,BrazilGov Name,BrazilGov Field ID,closest_asset_index,closest_asset_longitude,closest_asset_latitude,Unit Name,distance
0,0,-39.612,-21.360,ABALONE,1843.0,1,-39.612000,-21.360000,Abalone,0.000000
1,1,-38.076,-12.259,ACAJ�-BURIZINHO,1184.0,2,-38.076000,-12.259000,Acajá-Burizinho,0.000000
2,2,-37.366,-5.269,ACAU�,1380.0,3,-37.366000,-5.269000,Acauã,0.000000
3,3,-36.889,-10.686,AGUILHADA,1188.0,5,-36.889000,-10.686000,Aguilhada,0.000000
4,4,-36.262,-4.899,AGULHA,1189.0,6,-36.262000,-4.899000,Agulha,0.000000
...,...,...,...,...,...,...,...,...,...,...
422,422,-39.863,-21.649,WAHOO,2360.0,12323,-39.863330,-21.649060,Wahoo,0.000335
423,423,-39.035,-3.032,XAR�U,981.0,460,-39.035000,-3.032000,Xaréu,0.000000
424,424,-50.458,-25.804,XISTO S�O MATEUS DO SUL,2600.0,46,-52.005772,-24.719196,Barra Bonita,1.890079
425,425,-38.344,-12.371,�GUA GRANDE,1187.0,463,-38.344000,-12.371000,Água Grande,0.000000


In [82]:
# Save Brazil_to_goget_assets_df to csv
Brazil_to_goget_assets_df.to_csv(Comparison_folder + 'Brazil_to_goget_assets_df_' + timestamp + '.csv', index=False)